<a href="https://colab.research.google.com/github/VenkatesanNadimuthu/C3AN-Model/blob/001-gpt2-pretrain-recipes/Experiments/SLM_Recipe_Trainer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import tiktoken
import math
import time
import os
import re
import random
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import logging

# ==========================================
# 1. Configuration & Hardware Setup
# ==========================================

class Config:
    # --------------------------------------
    # Data Configuration
    # --------------------------------------
    dataset_path = '/content/structured_recipes_pretrain.txt'  # <-- SET YOUR DATASET PATH HERE

    # --------------------------------------
    # Model Architecture (SLM Size)
    # --------------------------------------
    block_size = 512       # Context length
    vocab_size = 50304     # GPT-2 vocab (50257) padded to nearest multiple of 64 for efficiency
    n_layer = 6            # Number of Transformer Blocks
    n_head = 8             # Number of Attention Heads
    n_embd = 512           # Embedding dimension
    dropout = 0.1

    # --------------------------------------
    # Training Hyperparameters
    # --------------------------------------
    batch_size = 16        # A100 allows larger batches, but we keep 16 for safety/speed balance
    learning_rate = 3e-4
    max_iters = 1000       # Short run for demonstration (Increase for convergence)
    eval_interval = 200    # How often to check metrics
    eval_iters = 50        # How many batches to use for PPL estimation

    # --------------------------------------
    # System
    # --------------------------------------
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    # Use bfloat16 if on Ampere (A100) or newer, else float16
    dtype = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16'

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Hardware Check & Setup
if Config.device == 'cuda':
    logger.info(f"Using GPU: {torch.cuda.get_device_name(0)}")
    logger.info(f"Compute Capability: {torch.cuda.get_device_capability(0)}")
    if Config.dtype == 'bfloat16':
        logger.info("A100/Ampere detected. Using bfloat16 for optimization.")
else:
    logger.warning("CUDA not available. Falling back to CPU. Training will be extremely slow.")

# Seed for reproducibility
torch.manual_seed(1337)
if Config.device == 'cuda':
    torch.cuda.manual_seed(1337)

# ==========================================
# 2. Data Pipeline & Tokenizer
# ==========================================

class RecipeDataset:
    def __init__(self, file_path, split='train'):
        self.file_path = file_path

        # Load Data
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"Dataset not found at {file_path}")

        with open(file_path, 'r', encoding='utf-8') as f:
            raw_text = f.read()

        # Using Tiktoken (BPE)
        logger.info("Tokenizing data with tiktoken (gpt2)...")
        enc = tiktoken.get_encoding("gpt2")
        self.enc = enc

        # We process the text as a continuous stream of tokens
        self.data = torch.tensor(enc.encode(raw_text), dtype=torch.long)

        # Validation: Ensure data is large enough for context window
        if len(self.data) <= Config.block_size:
            logger.warning(f"Dataset too small ({len(self.data)} tokens) for block_size {Config.block_size}. Duplicating data for stability.")
            repeat_factor = (Config.block_size // len(self.data)) + 2
            self.data = self.data.repeat(repeat_factor)

        # 90/10 Split
        n = int(0.9 * len(self.data))
        if split == 'train':
            self.data = self.data[:n]
        else:
            self.data = self.data[n:]

        # For RGA Evaluation: Parsing specific recipes from raw text for the test set
        if split == 'test':
            # Use the original text for parsing to avoid tokenization artifacts
            self.raw_recipes = self._parse_recipes(raw_text)

    def _parse_recipes(self, text_segment):
        """Helper to extract (Title, Ingredients) tuples for F1 eval."""
        # Splitting by [BOS] assuming the format in the file
        raw_items = text_segment.split('[BOS]')
        parsed = []
        for item in raw_items:
            # Simple regex logic to capture Title and Ingredients
            title_match = re.search(r'\*\*Title:\*\*\s*(.*?)\n', item)
            ing_match = re.search(r'\*\*Ingredients:\*\*(.*?)\*\*Instructions:\*\*', item, re.DOTALL)

            if title_match and ing_match:
                title = title_match.group(1).strip()
                ingredients = ing_match.group(1).strip()
                parsed.append({'title': title, 'ingredients': ingredients})
        return parsed

    def get_batch(self):
        # Ensure high is greater than low
        high = len(self.data) - Config.block_size
        if high <= 0:
            high = 1 # Fallback for extremely small validation sets

        ix = torch.randint(high, (Config.batch_size,))
        x = torch.stack([self.data[i:i+Config.block_size] for i in ix])
        y = torch.stack([self.data[i+1:i+Config.block_size+1] for i in ix])
        return x.to(Config.device), y.to(Config.device)

# ==========================================
# 3. Model Architecture (Decoder-Only Transformer)
# ==========================================

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=False)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=False)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.dropout = config.dropout
        # Flash Attention is handled functionally

    def forward(self, x):
        B, T, C = x.size()

        # Calculate query, key, values for all heads in batch
        q, k, v  = self.c_attn(x).split(self.n_embd, dim=2)

        # Reshape for multi-head attention: (B, T, n_head, head_size) -> (B, n_head, T, head_size)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        # F.scaled_dot_product_attention uses Flash Attention 2 if available on A100
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=self.dropout if self.training else 0, is_causal=True)

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.c_proj(y)
        return y

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc    = nn.Linear(config.n_embd, 4 * config.n_embd, bias=False)
        self.gelu    = nn.GELU()
        self.c_proj  = nn.Linear(4 * config.n_embd, config.n_embd, bias=False)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        x = self.dropout(x)
        return x

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class SLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict({
            'wte': nn.Embedding(config.vocab_size, config.n_embd),
            'wpe': nn.Embedding(config.block_size, config.n_embd),
            'drop': nn.Dropout(config.dropout),
            'h': nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            'ln_f': nn.LayerNorm(config.n_embd),
        })
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        # Weight tying
        self.transformer.wte.weight = self.lm_head.weight

        # Init weights
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        device = idx.device
        b, t = idx.size()
        pos = torch.arange(0, t, dtype=torch.long, device=device)

        tok_emb = self.transformer.wte(idx)
        pos_emb = self.transformer.wpe(pos)
        x = self.transformer.drop(tok_emb + pos_emb)

        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)

        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        else:
            logits = self.lm_head(x[:, [-1], :]) # optimize inference, only last token
            loss = None

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """
        Generation loop.
        """
        for _ in range(max_new_tokens):
            # crop context
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]

            # Forward
            logits, _ = self.forward(idx_cond)
            logits = logits[:, -1, :] / temperature

            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')

            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)

        return idx

# ==========================================
# 4. Evaluation Engine (PPL, RGA, Checks)
# ==========================================

class Evaluator:
    def __init__(self, model, test_dataset, enc):
        self.model = model
        self.dataset = test_dataset
        self.enc = enc

    @torch.no_grad()
    def calculate_perplexity(self):
        """Standard PPL calculation on Test Set."""
        self.model.eval()
        losses = torch.zeros(Config.eval_iters)
        for k in range(Config.eval_iters):
            X, Y = self.dataset.get_batch()
            with torch.amp.autocast(device_type=Config.device, dtype=getattr(torch, Config.dtype)):
                _, loss = self.model(X, Y)
            losses[k] = loss.item()
        mean_loss = losses.mean()
        ppl = torch.exp(mean_loss)
        return ppl.item(), mean_loss.item()

    def calculate_ingredient_f1(self, num_samples=20):
        """
        Recipe Generation Accuracy (RGA):
        1. Select 'num_samples' random recipes from test set.
        2. Prompt model with the Title.
        3. Extract generated ingredients.
        4. Compare to Ground Truth ingredients using F1 Score (Set of words).
        """
        self.model.eval()
        f1_scores = []

        valid_samples = [r for r in self.dataset.raw_recipes if len(r['ingredients']) > 5]
        if len(valid_samples) < num_samples:
            num_samples = len(valid_samples)

        if num_samples == 0:
            print("No valid samples found for RGA.")
            return 0.0

        samples = random.sample(valid_samples, num_samples)

        print(f"\n--- Running RGA (F1 Score) on {num_samples} samples ---")

        for i, sample in enumerate(samples):
            # Prompt Construction
            prompt_text = f"[BOS] \n **Title:** {sample['title']} \n"
            start_ids = self.enc.encode(prompt_text)
            x = torch.tensor(start_ids, dtype=torch.long, device=Config.device)[None, ...]

            # Generate
            # Limit generation to avoid extremely long waits, roughly enough for ingredients
            y = self.model.generate(x, max_new_tokens=150, temperature=0.8)
            generated_text = self.enc.decode(y[0].tolist())

            # Extraction Logic
            # We look for the section between Title and Instructions (or next section)
            gen_ing_match = re.search(r'\*\*Ingredients:\*\*(.*?)(?:\*\*Instructions|\Z)', generated_text, re.DOTALL)

            if gen_ing_match:
                gen_ingredients = gen_ing_match.group(1).lower()
                truth_ingredients = sample['ingredients'].lower()

                # Tokenize into words (sets)
                gen_set = set(re.findall(r'\w+', gen_ingredients))
                truth_set = set(re.findall(r'\w+', truth_ingredients))

                # F1 Calculation
                if len(gen_set) == 0:
                    f1 = 0.0
                else:
                    intersection = len(gen_set.intersection(truth_set))
                    precision = intersection / len(gen_set)
                    recall = intersection / len(truth_set)
                    if (precision + recall) > 0:
                        f1 = 2 * (precision * recall) / (precision + recall)
                    else:
                        f1 = 0.0
                f1_scores.append(f1)

                if i == 0: # Print one example for sanity check
                    print(f"Sample Title: {sample['title']}")
                    print(f"GT Ingredients len: {len(truth_set)} | Gen Ingredients len: {len(gen_set)}")
                    print(f"Sample F1: {f1:.4f}")
            else:
                f1_scores.append(0.0) # Failed to generate ingredients section

        avg_f1 = sum(f1_scores) / len(f1_scores) if f1_scores else 0
        return avg_f1

    def offline_sanity_check(self):
        """Asserts that inference runs without internet."""
        print("\n--- Running Offline Check ---")
        # In a real rigorous test, we would disable socket here.
        # For this script, we check if the model is a local torch object and runs forward.
        try:
            x = torch.zeros((1, 10), dtype=torch.long, device=Config.device)
            # Simple forward pass
            self.model(x)
            print("PASS: Model forward pass successful locally.")
            return True
        except Exception as e:
            print(f"FAIL: Model failed local execution. Error: {e}")
            return False

    def generate_human_eval_template(self):
        """Outputs a template for manual scoring."""
        template = """
        ===============================================================
        HUMAN EVALUATION FORM (IEEE Standard)
        ===============================================================
        Instruction: Rate the following generated recipe on 'Instructional Coherence'.
        Scale: 1 (Complete Nonsense) to 5 (Perfectly Logical & Safe).

        Evaluator 1 Score: [   ]
        Evaluator 2 Score: [   ]
        Evaluator 3 Score: [   ]

        Metric Definition:
        - Coherence: Do steps follow logically?
        - Halucination: Are ingredients mentioned in steps present in list?
        ===============================================================
        """
        print(template)

# ==========================================
# 5. Training Loop
# ==========================================

def train():
    # Load Data
    print("Loading datasets...")
    # NOTE: Ensure 'structured_recipes_pretrain.txt' is uploaded to Colab

    if not os.path.exists(Config.dataset_path):
        # Create dummy file if not exists for code to be runnable immediately in test envs
        print(f"Warning: Dataset not found at {Config.dataset_path}. Creating dummy data.")
        # Ensure dummy data is substantial enough for block_size=512
        dummy_content = "[BOS] \n **Title:** Dummy Recipe \n **Ingredients:** Water \n **Instructions:** Drink. [BOS] " * 100
        with open(Config.dataset_path, 'w') as f:
            f.write(dummy_content)

    train_dataset = RecipeDataset(Config.dataset_path, split='train')
    test_dataset = RecipeDataset(Config.dataset_path, split='test')

    # Init Model
    model = SLM(Config).to(Config.device)
    print(f"Model Parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")

    # Optimizer
    optimizer = torch.optim.AdamW(model.parameters(), lr=Config.learning_rate)

    # AMP Scaler for A100/Float16
    # UPDATED: Using unified torch.amp.GradScaler (PyTorch 2.4+) instead of legacy torch.cuda.amp
    scaler = torch.amp.GradScaler(device=Config.device, enabled=(Config.dtype != 'float32'))

    # Metrics Storage
    loss_history = []
    ppl_history = []

    # Evaluator
    evaluator = Evaluator(model, test_dataset, train_dataset.enc)

    print("Starting Training...")
    start_time = time.time()

    model.train()

    for iter in range(Config.max_iters):

        # Periodic Evaluation
        if iter % Config.eval_interval == 0 or iter == Config.max_iters - 1:
            ppl, val_loss = evaluator.calculate_perplexity()
            ppl_history.append(ppl)
            print(f"Iter {iter}: Train Loss {loss_history[-1] if loss_history else 0:.4f} | Val PPL {ppl:.2f}")
            model.train()

        # Training Step
        xb, yb = train_dataset.get_batch()

        # Mixed Precision Context
        # 'device_type' must be 'cuda' or 'cpu', 'dtype' handles bfloat16 vs float16
        pt_dtype = getattr(torch, Config.dtype)
        with torch.amp.autocast(device_type=Config.device, dtype=pt_dtype):
            logits, loss = model(xb, yb)

        # Backward Pass
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        loss_history.append(loss.item())

    total_time = time.time() - start_time
    print(f"Training finished in {total_time:.2f} seconds.")

    # ==========================================
    # 6. Final Evaluation & Visualization
    # ==========================================

    # 1. Domain Accuracy (PPL)
    final_ppl, _ = evaluator.calculate_perplexity()
    print(f"Final Test Perplexity: {final_ppl:.2f}")

    # 2. Recipe Generation Accuracy (F1)
    f1_score = evaluator.calculate_ingredient_f1(num_samples=20)
    print(f"Recipe Generation Ingredient F1 Score: {f1_score:.4f}")

    # 3. Offline Check
    evaluator.offline_sanity_check()

    # 4. Human Eval
    evaluator.generate_human_eval_template()

    # Visualization
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(loss_history)
    plt.title("Training Loss")
    plt.xlabel("Iteration")

    plt.subplot(1, 2, 2)
    plt.plot(range(0, Config.max_iters + 1, Config.eval_interval)[:-1] if len(ppl_history) < len(range(0, Config.max_iters, Config.eval_interval)) else range(0, Config.max_iters + 1, Config.eval_interval)[:len(ppl_history)], ppl_history)
    plt.title("Validation Perplexity")
    plt.xlabel("Iteration")
    plt.tight_layout()
    plt.savefig("training_metrics.png")
    plt.close() # Clean up memory
    print("Metrics saved to training_metrics.png")

    return model, train_dataset.enc

# ==========================================
# 7. Interactive Inference Loop
# ==========================================

def interactive_inference(model, enc):
    print("\n" + "="*40)
    print(" INTERACTIVE RECIPE GENERATOR")
    print(" Enter a Recipe Title (or 'q' to quit)")
    print("="*40)

    model.eval()
    while True:
        user_input = input("\nRecipe Title: ")
        if user_input.lower() == 'q':
            break

        # Prompt Engineering specific to the dataset format
        prompt = f"[BOS] \n **Title:** {user_input} \n"

        start_ids = enc.encode(prompt)
        x = torch.tensor(start_ids, dtype=torch.long, device=Config.device)[None, ...]

        print("\nGenerating...", end="", flush=True)
        with torch.no_grad():
            # Generate 300 tokens max for a full recipe
            y = model.generate(x, max_new_tokens=300, temperature=0.8, top_k=50)

        output_text = enc.decode(y[0].tolist())
        print("\n" + "-"*40)
        print(output_text)
        print("-"*40)

if __name__ == "__main__":
    # Ensure matplotlib doesn't try to open window in headless env
    import matplotlib
    matplotlib.use('Agg')

    trained_model, tokenizer = train()

    # Switch backend back for interactive if supported (optional)
    # Start Interactive Mode
    interactive_inference(trained_model, tokenizer)

Loading datasets...
Model Parameters: 44.91M
Starting Training...
Iter 0: Train Loss 0.0000 | Val PPL 57767.68
Iter 200: Train Loss 2.6257 | Val PPL 14.99
Iter 400: Train Loss 2.4102 | Val PPL 9.55
Iter 600: Train Loss 1.9888 | Val PPL 7.33
Iter 800: Train Loss 1.7728 | Val PPL 6.46
Iter 999: Train Loss 1.7375 | Val PPL 5.87
Training finished in 33.87 seconds.
Final Test Perplexity: 5.89
No valid samples found for RGA.
Recipe Generation Ingredient F1 Score: 0.0000

--- Running Offline Check ---
PASS: Model forward pass successful locally.

        HUMAN EVALUATION FORM (IEEE Standard)
        Instruction: Rate the following generated recipe on 'Instructional Coherence'.
        Scale: 1 (Complete Nonsense) to 5 (Perfectly Logical & Safe).
        
        Evaluator 1 Score: [   ]
        Evaluator 2 Score: [   ]
        Evaluator 3 Score: [   ]
        
        Metric Definition:
        - Coherence: Do steps follow logically?
        - Halucination: Are ingredients mentioned in steps 